# KCDK trash-talk fact engine
This notebook demonstrates deterministic factual candidates and selection. It does not generate jokes, call OpenAI, or post to Discord.

In [ ]:
from pathlib import Path
import json
import sys
import tempfile

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from kcdk.facts import build_weekly_fact_report
from kcdk.persistence import connect_database, import_week

## Import the fictional season into a temporary database

In [ ]:
temporary_directory_context = tempfile.TemporaryDirectory(prefix='kcdk-facts-')
database_path = Path(temporary_directory_context.name) / 'facts.sqlite'
connection = connect_database(database_path)
mock_directory = ROOT / 'data' / 'mock'

summaries = [
    import_week(
        connection,
        mock_directory / f'season_week_{week}.csv',
        mock_directory / 'members.csv',
        season_name='Mock 2026',
        season_identifier='mock-2026',
        season_year=2026,
        week_number=week,
        contest_name=f'Fictional Week {week}',
        contest_date=f'2026-09-{week:02d}',
    ).to_dict()
    for week in range(1, 5)
]
summaries

## Build the latest-week report

In [ ]:
report = build_weekly_fact_report(connection, 'mock-2026', max_facts=12)
{
    'season': report.season_name,
    'week': report.week_label,
    'candidate_count': report.generated_candidate_count,
    'selected_count': len(report.selected_facts),
}

## All candidate facts, priorities, and tags

In [ ]:
candidate_rows = [
    {
        'priority': fact.priority,
        'category': fact.category,
        'fact_type': fact.fact_type,
        'subject': fact.subject_member_name,
        'related_member': fact.related_member_name,
        'player': fact.related_player,
        'completeness': fact.completeness,
        'tags': ', '.join(fact.tags),
        'summary': fact.summary,
    }
    for fact in report.candidate_facts
]
pd.DataFrame(candidate_rows)

## Balanced deterministic shortlist

In [ ]:
pd.DataFrame([fact.to_dict() for fact in report.selected_facts])[
    ['priority', 'category', 'fact_type', 'subject_member_name', 'tags', 'summary']
]

## JSON payload and completeness warnings

In [ ]:
payload = json.loads(report.to_json())
payload

In [ ]:
report.warnings

In [ ]:
connection.close()
temporary_directory_context.cleanup()
{'database_cleaned_up': not database_path.exists()}